# Slack Alert — Pipeline Completion

Custom task triggered by `forge deploy` after `customer_summary` completes.  
Sends a Slack notification with pipeline status, environment, and timestamp.

**Referenced in `forge.yml`:**
```yaml
custom_tasks:
  - task_key: notify_slack
    notebook_task: notebooks/slack_alert.ipynb
    stage: serve
    depends_on: [customer_summary]
```

In [ ]:
# Read pipeline context from Databricks widgets
dbutils.widgets.text("pipeline_name", "PROCESS_demo")
dbutils.widgets.text("status", "SUCCESS")
dbutils.widgets.text("environment", "dev")

pipeline_name = dbutils.widgets.get("pipeline_name")
status = dbutils.widgets.get("status")
environment = dbutils.widgets.get("environment")

In [ ]:
import json
from datetime import datetime, timezone

# Build Slack message payload
emoji = ":white_check_mark:" if status == "SUCCESS" else ":x:"
color = "#36a64f" if status == "SUCCESS" else "#dc3545"

payload = {
    "attachments": [{
        "color": color,
        "blocks": [
            {
                "type": "section",
                "text": {
                    "type": "mrkdwn",
                    "text": (
                        f"{emoji} *Pipeline {status}*\n"
                        f"*Pipeline:* `{pipeline_name}`\n"
                        f"*Environment:* `{environment}`\n"
                        f"*Timestamp:* {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}"
                    ),
                },
            }
        ],
    }]
}

print(json.dumps(payload, indent=2))

In [ ]:
import urllib.request

# Send to Slack (webhook URL stored in Databricks secrets)
webhook_url = dbutils.secrets.get(scope="forge", key="slack_webhook_url")

req = urllib.request.Request(
    webhook_url,
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req) as resp:
    print(f"Slack response: {resp.status} {resp.read().decode()}")